# Seq2Seq注意力与多头注意力

本notebook介绍如何将注意力机制应用于序列到序列模型,以及多头注意力的设计与实现。

## 学习目标

- 理解Bahdanau注意力在机器翻译中的应用
- 实现带注意力的编码器-解码器模型
- 掌握多头注意力的原理与并行计算
- 可视化注意力权重矩阵

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from collections import Counter

torch.manual_seed(42)

## 1. Seq2Seq模型的问题

回顾第8章的seq2seq模型:

### 1.1 架构回顾

**编码器**: 将变长输入序列压缩为固定长度的上下文向量
$$
\mathbf{c} = q(\mathbf{h}_1, \ldots, \mathbf{h}_T)
$$

通常取最后一个隐状态: $\mathbf{c} = \mathbf{h}_T$

**解码器**: 根据固定上下文 $\mathbf{c}$ 生成输出序列
$$
\mathbf{s}_{t'} = g(\mathbf{y}_{t'-1}, \mathbf{c}, \mathbf{s}_{t'-1})
$$

### 1.2 信息瓶颈问题

**问题**: 所有输入信息都压缩到单个固定维度的向量 $\mathbf{c}$!

**后果**:
1. **长序列性能下降**: 长句子的信息难以完全编码
2. **均匀关注**: 解码每个词时都使用相同的上下文
3. **梯度消失**: 编码器开头的信息传递困难

**例子**: 英译法
- 输入: "The cat sat on the mat"
- 翻译 "chat" 时应关注 "cat"
- 翻译 "tapis" 时应关注 "mat"
- 但固定上下文 $\mathbf{c}$ 无法区分!

## 2. Bahdanau注意力机制

**关键思想**: 让每个解码步使用**不同的上下文向量** $\mathbf{c}_{t'}$!

### 2.1 动态上下文向量

在解码时间步 $t'$,上下文向量为:

$$
\mathbf{c}_{t'} = \sum_{t=1}^T \alpha_{t't} \mathbf{h}_t
$$

其中:
- $\mathbf{h}_t$: 编码器在时间 $t$ 的隐状态(键和值)
- $\mathbf{s}_{t'-1}$: 解码器在时间 $t'-1$ 的隐状态(查询)
- $\alpha_{t't}$: 注意力权重

### 2.2 注意力权重计算

**评分函数**(加性注意力):
$$
e_{t't} = a(\mathbf{s}_{t'-1}, \mathbf{h}_t) = \mathbf{v}^\top \tanh(\mathbf{W}_s \mathbf{s}_{t'-1} + \mathbf{W}_h \mathbf{h}_t)
$$

**归一化**(softmax):
$$
\alpha_{t't} = \frac{\exp(e_{t't})}{\sum_{k=1}^T \exp(e_{t'k})}
$$

### 2.3 解码器计算流程

在时间步 $t'$:
1. 使用 $\mathbf{s}_{t'-1}$ 作为查询计算注意力权重 $\alpha_{t't}$
2. 计算上下文向量 $\mathbf{c}_{t'} = \sum_t \alpha_{t't} \mathbf{h}_t$
3. 拼接输入嵌入和上下文: $[\mathbf{y}_{t'-1}; \mathbf{c}_{t'}]$
4. 更新隐状态: $\mathbf{s}_{t'} = g([\mathbf{y}_{t'-1}; \mathbf{c}_{t'}], \mathbf{s}_{t'-1})$
5. 预测输出: $P(y_{t'} | y_{<t'}, \mathbf{x}) = \text{softmax}(\mathbf{W}_o \mathbf{s}_{t'})$

In [ ]:
# 复用第1个notebook的AdditiveAttention
class AdditiveAttention(nn.Module):
    """加性注意力(Bahdanau注意力)"""
    def __init__(self, key_size, query_size, num_hiddens, dropout=0.1):
        super().__init__()
        self.W_k = nn.Linear(key_size, num_hiddens, bias=False)
        self.W_q = nn.Linear(query_size, num_hiddens, bias=False)
        self.w_v = nn.Linear(num_hiddens, 1, bias=False)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, queries, keys, values, valid_lens=None):
        queries = self.W_q(queries)
        keys = self.W_k(keys)
        features = queries.unsqueeze(2) + keys.unsqueeze(1)
        features = torch.tanh(features)
        scores = self.w_v(features).squeeze(-1)
        self.attention_weights = self.masked_softmax(scores, valid_lens)
        return torch.bmm(self.dropout(self.attention_weights), values)
    
    def masked_softmax(self, X, valid_lens):
        if valid_lens is None:
            return F.softmax(X, dim=-1)
        shape = X.shape
        if valid_lens.dim() == 1:
            valid_lens = valid_lens.repeat_interleave(shape[1])
        else:
            valid_lens = valid_lens.reshape(-1)
        X = X.reshape(-1, shape[-1])
        mask = torch.arange(X.shape[1], device=X.device)[None, :] < valid_lens[:, None]
        X[~mask] = -1e6
        return F.softmax(X.reshape(shape), dim=-1)

In [ ]:
class Seq2SeqEncoder(nn.Module):
    """RNN编码器(与第8章相同)"""
    def __init__(self, vocab_size, embed_size, num_hiddens, num_layers, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.rnn = nn.GRU(embed_size, num_hiddens, num_layers, dropout=dropout)
        
    def forward(self, X, state=None):
        """
        X: (batch_size, num_steps)
        返回: outputs (batch_size, num_steps, num_hiddens), state
        """
        # X: (batch_size, num_steps, embed_size)
        X = self.embedding(X)
        # 转换为RNN输入格式: (num_steps, batch_size, embed_size)
        X = X.permute(1, 0, 2)
        # outputs: (num_steps, batch_size, num_hiddens)
        outputs, state = self.rnn(X, state)
        # 转回: (batch_size, num_steps, num_hiddens)
        return outputs.permute(1, 0, 2), state

class Seq2SeqAttentionDecoder(nn.Module):
    """带Bahdanau注意力的RNN解码器"""
    def __init__(self, vocab_size, embed_size, num_hiddens, num_layers, dropout=0.1):
        super().__init__()
        self.attention = AdditiveAttention(num_hiddens, num_hiddens, num_hiddens, dropout)
        self.embedding = nn.Embedding(vocab_size, embed_size)
        # 输入是[嵌入; 上下文],所以输入维度是embed_size + num_hiddens
        self.rnn = nn.GRU(embed_size + num_hiddens, num_hiddens, num_layers, dropout=dropout)
        self.dense = nn.Linear(num_hiddens, vocab_size)
        
    def init_state(self, enc_outputs, enc_valid_lens):
        """
        enc_outputs: (outputs, state)
            outputs: (batch_size, num_steps, num_hiddens)
            state: (num_layers, batch_size, num_hiddens)
        """
        outputs, hidden_state = enc_outputs
        return (outputs, hidden_state, enc_valid_lens)
    
    def forward(self, X, state):
        """
        X: (batch_size, num_steps)
        state: (enc_outputs, hidden_state, enc_valid_lens)
        """
        enc_outputs, hidden_state, enc_valid_lens = state
        # X: (batch_size, num_steps, embed_size)
        X = self.embedding(X)
        # 转换为RNN格式: (num_steps, batch_size, embed_size)
        X = X.permute(1, 0, 2)
        
        outputs, self._attention_weights = [], []
        for x in X:
            # query: 上一时间步的解码器隐状态 (batch_size, 1, num_hiddens)
            query = hidden_state[-1].unsqueeze(1)
            # context: (batch_size, 1, num_hiddens)
            context = self.attention(query, enc_outputs, enc_outputs, enc_valid_lens)
            # 拼接嵌入和上下文: (batch_size, 1, embed_size + num_hiddens)
            x = torch.cat([context, x.unsqueeze(1)], dim=-1)
            # 转换为RNN输入: (1, batch_size, embed_size + num_hiddens)
            out, hidden_state = self.rnn(x.permute(1, 0, 2), hidden_state)
            outputs.append(out)
            self._attention_weights.append(self.attention.attention_weights)
        
        # outputs: (num_steps, batch_size, num_hiddens)
        outputs = torch.cat(outputs, dim=0)
        # 全连接层: (num_steps, batch_size, vocab_size)
        outputs = self.dense(outputs)
        # 转回: (batch_size, num_steps, vocab_size)
        return outputs.permute(1, 0, 2), [enc_outputs, hidden_state, enc_valid_lens]
    
    @property
    def attention_weights(self):
        return self._attention_weights

# 测试模型结构
batch_size, num_steps = 4, 7
vocab_size, embed_size, num_hiddens, num_layers = 10, 8, 16, 2

encoder = Seq2SeqEncoder(vocab_size, embed_size, num_hiddens, num_layers)
decoder = Seq2SeqAttentionDecoder(vocab_size, embed_size, num_hiddens, num_layers)

X = torch.zeros((batch_size, num_steps), dtype=torch.long)
enc_outputs = encoder(X)
state = decoder.init_state(enc_outputs, None)
output, state = decoder(X, state)

print(f"输出形状: {output.shape}")  # (4, 7, 10)
print(f"注意力权重数量: {len(decoder.attention_weights)}")  # 7个时间步
print(f"每个权重形状: {decoder.attention_weights[0].shape}")  # (4, 1, 7)

## 3. 玩具翻译实验

用简单的数字序列模拟翻译任务,更清晰地展示注意力权重。

**任务**: 将升序序列反转
- 输入: [1, 2, 3, 4, 5]
- 输出: [5, 4, 3, 2, 1]

**期望**: 注意力权重矩阵应呈反对角线模式!

In [ ]:
# 生成简单数据集
def generate_reverse_data(num_samples=1000, max_len=10, vocab_size=20):
    """生成序列反转数据"""
    data = []
    for _ in range(num_samples):
        length = np.random.randint(3, max_len + 1)
        seq = np.random.randint(1, vocab_size, size=length)
        # 输入序列和反转的输出序列
        src = seq.tolist()
        tgt = seq[::-1].tolist()
        data.append((src, tgt))
    return data

def pad_sequences(sequences, pad_id=0):
    """填充序列到相同长度"""
    max_len = max(len(seq) for seq in sequences)
    padded = [seq + [pad_id] * (max_len - len(seq)) for seq in sequences]
    lengths = [len(seq) for seq in sequences]
    return torch.tensor(padded), torch.tensor(lengths)

class ReverseDataset(torch.utils.data.Dataset):
    def __init__(self, data):
        self.data = data
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]

def collate_fn(batch):
    """批处理函数"""
    src_seqs = [item[0] for item in batch]
    tgt_seqs = [item[1] for item in batch]
    
    src_padded, src_lengths = pad_sequences(src_seqs)
    tgt_padded, tgt_lengths = pad_sequences(tgt_seqs)
    
    return src_padded, src_lengths, tgt_padded, tgt_lengths

# 生成数据
train_data = generate_reverse_data(num_samples=1000, max_len=8, vocab_size=15)
test_data = generate_reverse_data(num_samples=100, max_len=8, vocab_size=15)

train_dataset = ReverseDataset(train_data)
train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn
)

print(f"训练样本: {len(train_data)}")
print(f"示例: {train_data[0]}")

In [ ]:
# 训练函数
def train_epoch(encoder, decoder, data_loader, optimizer, criterion):
    encoder.train()
    decoder.train()
    total_loss = 0
    
    for src, src_len, tgt, tgt_len in data_loader:
        optimizer.zero_grad()
        
        # 编码
        enc_outputs = encoder(src)
        dec_state = decoder.init_state(enc_outputs, src_len)
        
        # 解码(teacher forcing)
        # 输入是目标序列向右移动一位
        dec_input = torch.cat([torch.zeros(tgt.size(0), 1, dtype=torch.long), tgt[:, :-1]], dim=1)
        dec_output, _ = decoder(dec_input, dec_state)
        
        # 计算损失(忽略填充位置)
        loss = 0
        for i in range(tgt.size(0)):
            valid_len = tgt_len[i]
            loss += criterion(dec_output[i, :valid_len], tgt[i, :valid_len])
        loss = loss / tgt.size(0)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(data_loader)

# 训练模型
vocab_size = 20  # 0是填充,1-15是词汇
embed_size, num_hiddens, num_layers = 16, 32, 1

encoder = Seq2SeqEncoder(vocab_size, embed_size, num_hiddens, num_layers)
decoder = Seq2SeqAttentionDecoder(vocab_size, embed_size, num_hiddens, num_layers)

optimizer = torch.optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=0.001)
criterion = nn.CrossEntropyLoss()

num_epochs = 20
for epoch in range(num_epochs):
    loss = train_epoch(encoder, decoder, train_loader, optimizer, criterion)
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {loss:.4f}")

print("\n训练完成!")

In [ ]:
# 推理和可视化注意力
def translate_and_visualize(encoder, decoder, src_seq):
    """翻译序列并可视化注意力权重"""
    encoder.eval()
    decoder.eval()
    
    with torch.no_grad():
        # 准备输入
        src = torch.tensor([src_seq])
        src_len = torch.tensor([len(src_seq)])
        
        # 编码
        enc_outputs = encoder(src)
        dec_state = decoder.init_state(enc_outputs, src_len)
        
        # 逐步解码
        dec_input = torch.zeros(1, 1, dtype=torch.long)
        outputs = []
        attention_weights_list = []
        
        for _ in range(len(src_seq)):
            dec_output, dec_state = decoder(dec_input, dec_state)
            pred = dec_output.argmax(dim=-1)
            outputs.append(pred.item())
            
            # 收集注意力权重
            attention_weights_list.append(decoder.attention_weights[-1].squeeze())
            
            # 下一个输入是当前预测
            dec_input = pred
        
        # 拼接注意力权重: (tgt_len, src_len)
        attention_matrix = torch.stack(attention_weights_list).numpy()
        
    return outputs, attention_matrix

# 测试几个样本
test_samples = [
    [1, 2, 3, 4, 5],
    [3, 7, 2, 9, 1, 6],
    [5, 8, 3],
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, src_seq in enumerate(test_samples):
    outputs, attention_matrix = translate_and_visualize(encoder, decoder, src_seq)
    
    ax = axes[i]
    sns.heatmap(attention_matrix, annot=True, fmt='.2f', cmap='YlOrRd', ax=ax,
                xticklabels=src_seq, yticklabels=outputs, cbar=True)
    ax.set_xlabel('Source (编码器)')
    ax.set_ylabel('Target (解码器)')
    ax.set_title(f'输入: {src_seq}\n输出: {outputs}\n真实: {src_seq[::-1]}')

plt.tight_layout()
plt.show()

print("\n注意力模式分析:")
print("- 对角线: 顺序翻译(如复制任务)")
print("- 反对角线: 逆序翻译(如反转任务) ← 期望看到的!")
print("- 其他模式: 更复杂的对齐关系")

## 4. 多头注意力(Multi-Head Attention)

**问题**: 单个注意力头只能学到一种对齐模式!

**解决方案**: 使用多个注意力头,每个学习不同的子空间表示。

### 4.1 多头注意力架构

给定查询 $\mathbf{q} \in \mathbb{R}^{d}$, 键 $\mathbf{k} \in \mathbb{R}^{d}$, 值 $\mathbf{v} \in \mathbb{R}^{d}$:

**对每个头 $i = 1, \ldots, h$**:
1. **线性投影**到子空间:
   $$
   \mathbf{q}_i = \mathbf{W}_i^{(q)} \mathbf{q}, \quad \mathbf{k}_i = \mathbf{W}_i^{(k)} \mathbf{k}, \quad \mathbf{v}_i = \mathbf{W}_i^{(v)} \mathbf{v}
   $$
   其中 $\mathbf{W}_i^{(q)}, \mathbf{W}_i^{(k)}, \mathbf{W}_i^{(v)} \in \mathbb{R}^{(d/h) \times d}$

2. **计算单头注意力**:
   $$
   \mathbf{head}_i = \text{Attention}(\mathbf{q}_i, \mathbf{k}_i, \mathbf{v}_i) \in \mathbb{R}^{d/h}
   $$

3. **拼接所有头**:
   $$
   \mathbf{head} = [\mathbf{head}_1; \ldots; \mathbf{head}_h] \in \mathbb{R}^{d}
   $$

4. **输出投影**:
   $$
   \text{MultiHead}(\mathbf{q}, \mathbf{k}, \mathbf{v}) = \mathbf{W}^{(o)} \mathbf{head}
   $$
   其中 $\mathbf{W}^{(o)} \in \mathbb{R}^{d \times d}$

### 4.2 优势

1. **多种依赖关系**: 不同头关注不同的模式
   - Head 1: 短距离依赖(相邻词)
   - Head 2: 长距离依赖(句法关系)
   - Head 3: 语义关系

2. **表示能力**: 增强模型的表达能力

3. **并行计算**: 所有头可以同时计算

### 4.3 参数量

设 $d$ 是模型维度, $h$ 是头数:
- 单头注意力: $3d^2$ (Q, K, V投影)
- 多头注意力: $4d^2$ (Q, K, V投影 + 输出投影)
- 参数量相近,但多头性能更好!

In [ ]:
class DotProductAttention(nn.Module):
    """缩放点积注意力"""
    def __init__(self, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, queries, keys, values, valid_lens=None):
        d = queries.shape[-1]
        scores = torch.bmm(queries, keys.transpose(1, 2)) / np.sqrt(d)
        self.attention_weights = self.masked_softmax(scores, valid_lens)
        return torch.bmm(self.dropout(self.attention_weights), values)
    
    def masked_softmax(self, X, valid_lens):
        if valid_lens is None:
            return F.softmax(X, dim=-1)
        shape = X.shape
        if valid_lens.dim() == 1:
            valid_lens = valid_lens.repeat_interleave(shape[1])
        else:
            valid_lens = valid_lens.reshape(-1)
        X = X.reshape(-1, shape[-1])
        mask = torch.arange(X.shape[1], device=X.device)[None, :] < valid_lens[:, None]
        X[~mask] = -1e6
        return F.softmax(X.reshape(shape), dim=-1)

class MultiHeadAttention(nn.Module):
    """多头注意力"""
    def __init__(self, key_size, query_size, value_size, num_hiddens, num_heads, dropout=0.1):
        super().__init__()
        self.num_heads = num_heads
        self.attention = DotProductAttention(dropout)
        
        # 线性投影层
        self.W_q = nn.Linear(query_size, num_hiddens, bias=False)
        self.W_k = nn.Linear(key_size, num_hiddens, bias=False)
        self.W_v = nn.Linear(value_size, num_hiddens, bias=False)
        self.W_o = nn.Linear(num_hiddens, num_hiddens, bias=False)
        
    def forward(self, queries, keys, values, valid_lens=None):
        """
        queries: (batch_size, num_queries, query_size)
        keys: (batch_size, num_keys, key_size)
        values: (batch_size, num_keys, value_size)
        """
        # 线性投影: (batch_size, num_queries/keys, num_hiddens)
        queries = self.transpose_qkv(self.W_q(queries))
        keys = self.transpose_qkv(self.W_k(keys))
        values = self.transpose_qkv(self.W_v(values))
        
        # 调整valid_lens用于多头
        if valid_lens is not None:
            valid_lens = valid_lens.repeat_interleave(self.num_heads, dim=0)
        
        # 多头注意力: (batch_size*num_heads, num_queries, num_hiddens/num_heads)
        output = self.attention(queries, keys, values, valid_lens)
        
        # 恢复形状并输出投影: (batch_size, num_queries, num_hiddens)
        output_concat = self.transpose_output(output)
        return self.W_o(output_concat)
    
    def transpose_qkv(self, X):
        """变换形状以便多头并行计算"""
        # X: (batch_size, num_items, num_hiddens)
        # 分成多头: (batch_size, num_items, num_heads, num_hiddens/num_heads)
        X = X.reshape(X.shape[0], X.shape[1], self.num_heads, -1)
        # 交换维度: (batch_size, num_heads, num_items, num_hiddens/num_heads)
        X = X.permute(0, 2, 1, 3)
        # 合并batch和heads: (batch_size*num_heads, num_items, num_hiddens/num_heads)
        return X.reshape(-1, X.shape[2], X.shape[3])
    
    def transpose_output(self, X):
        """逆转transpose_qkv的操作"""
        # X: (batch_size*num_heads, num_items, num_hiddens/num_heads)
        X = X.reshape(-1, self.num_heads, X.shape[1], X.shape[2])
        X = X.permute(0, 2, 1, 3)
        return X.reshape(X.shape[0], X.shape[1], -1)

# 测试多头注意力
batch_size, num_queries, num_keys = 2, 4, 6
num_hiddens, num_heads = 100, 5

mha = MultiHeadAttention(num_hiddens, num_hiddens, num_hiddens, num_hiddens, num_heads)

X = torch.ones(batch_size, num_queries, num_hiddens)
Y = torch.ones(batch_size, num_keys, num_hiddens)
valid_lens = torch.tensor([3, 2])

output = mha(X, Y, Y, valid_lens)
print(f"多头注意力输出形状: {output.shape}")  # (2, 4, 100)
print(f"参数量: {sum(p.numel() for p in mha.parameters())}")

## 5. 可视化多头注意力

用简单的自注意力任务展示不同头学到的模式。

In [ ]:
# 创建合成数据:位置编码序列
def create_position_encoded_sequence(seq_len, d_model):
    """创建带位置编码的序列"""
    position = torch.arange(seq_len).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2) * (-np.log(10000.0) / d_model))
    pe = torch.zeros(seq_len, d_model)
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe

seq_len, d_model, num_heads = 10, 64, 4
X = create_position_encoded_sequence(seq_len, d_model).unsqueeze(0)

# 训练多头自注意力
mha = MultiHeadAttention(d_model, d_model, d_model, d_model, num_heads)
mha.eval()

with torch.no_grad():
    output = mha(X, X, X)
    # 获取每个头的注意力权重
    attention_weights = mha.attention.attention_weights.reshape(
        num_heads, seq_len, seq_len
    ).numpy()

# 可视化每个头的注意力模式
fig, axes = plt.subplots(1, num_heads, figsize=(16, 4))

for i in range(num_heads):
    ax = axes[i]
    sns.heatmap(attention_weights[i], annot=True, fmt='.2f', cmap='Blues', ax=ax,
                cbar=True, square=True)
    ax.set_title(f'Head {i+1}')
    ax.set_xlabel('Key Position')
    ax.set_ylabel('Query Position')

plt.tight_layout()
plt.show()

print("\n不同头的注意力模式:")
print("- Head 1: 可能关注对角线(局部上下文)")
print("- Head 2: 可能关注开头/结尾(全局信息)")
print("- Head 3: 可能关注特定距离(固定偏移)")
print("- Head 4: 可能学习其他模式")

## 6. 小结

### Bahdanau注意力

1. **动态上下文**: 每个解码步使用不同的上下文向量 $\mathbf{c}_{t'}$
2. **对齐学习**: 注意力权重自动学习源和目标的对齐
3. **性能提升**: 在长序列翻译中显著优于固定上下文
4. **可解释性**: 注意力权重可视化揭示模型决策过程

### 多头注意力

1. **多子空间**: 每个头在不同子空间学习
2. **多模式**: 捕获多种依赖关系(局部/全局,语法/语义)
3. **并行高效**: 所有头并行计算,无额外时间开销
4. **参数效率**: 与单头相比参数量仅增加1倍,但性能提升显著

### 关键公式对比

| 机制 | 评分函数 | 复杂度 | 用途 |
|------|----------|--------|------|
| 加性注意力 | $\mathbf{w}^\top \tanh(\mathbf{W}_q q + \mathbf{W}_k k)$ | $O(d^2)$ | Seq2seq |
| 点积注意力 | $(q^\top k) / \sqrt{d}$ | $O(d)$ | Transformer |
| 多头注意力 | $h$ 个点积注意力 | $O(hd)$ | Transformer |

### 下一步

- **自注意力**: 序列自己关注自己
- **位置编码**: 注入顺序信息
- **Transformer**: 完全基于注意力的架构

## 练习

1. **加性 vs 点积**: 在反转任务上比较两种评分函数的训练速度和最终性能。

2. **头数实验**: 尝试不同的头数(2, 4, 8, 16),观察性能和训练时间的权衡。

3. **注意力剪枝**: 训练后,尝试移除注意力权重最小的头,测试对性能的影响。

4. **可视化分析**: 在真实翻译任务上,分析哪些词对齐关系被不同的头捕获。

5. **长序列**: 测试注意力机制在不同序列长度(10, 50, 100, 500)上的性能变化。